# Vision Transformer (ViT) Document Classification - M4 MacBook Optimized 🚀

**Optimized for Apple Silicon M4 10-core MacBook with Authenticator.ai image dataset**

This notebook trains a Vision Transformer for document image classification with M4-specific optimizations:

- **4 Categories**: Education, Insurance, Legal, Resume/Employment (2,000 image samples)
- **16 Document Types**: Various subtypes across categories  
- **Apple Silicon MPS**: Optimized for M4 GPU acceleration
- **Memory Efficient**: Batch size and image processing optimized for M4
- **Performance Tuned**: 10-core CPU utilization, optimized data loading
- **MVP Ready**: Direct integration with your cleaned image dataset

## M4 Optimizations:
- MPS (Metal Performance Shaders) acceleration for image processing
- Optimized image transforms and data augmentation
- Memory-efficient ViT architecture
- Gradient accumulation for effective training
- Early stopping and learning rate scheduling

In [ ]:
# Install M4-optimized packages for Vision Transformer
%pip install torch torchvision torchaudio --quiet
%pip install transformers timm datasets --quiet
%pip install scikit-learn evaluate accelerate --quiet
%pip install pillow psutil matplotlib seaborn --quiet


In [ ]:
import os
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from transformers import (
    ViTImageProcessor, 
    ViTForImageClassification,
    get_linear_schedule_with_warmup,
    logging as hf_logging
)

import psutil
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("🍎 Authenticator.ai - ViT MVP Training (M4 Optimized)")
print("=" * 60)


In [ ]:
# M4 Device Setup for Vision Tasks
def setup_m4_device():
    if torch.backends.mps.is_available():
        device = 'mps'
        print("🚀 Using Apple Silicon MPS acceleration for ViT")
        torch.backends.mps.empty_cache()
    elif torch.cuda.is_available():
        device = 'cuda'
        print("🚀 Using CUDA acceleration")
    else:
        device = 'cpu'
        print("💻 Using CPU")
    return device

DEVICE = setup_m4_device()

# System info optimized for image processing
cpu_count = psutil.cpu_count()
memory = psutil.virtual_memory()
print(f"🖥️  M4 System: {cpu_count} cores, {memory.total // (1024**3)} GB RAM")
print(f"📱 PyTorch: {torch.__version__}, MPS: {torch.backends.mps.is_available()}")
print(f"🎯 Device: {DEVICE}")
print(f"🖼️  Ready for image processing!")

DEVICE


In [ ]:
# M4-Optimized Configuration for ViT Image Classification
class M4ViTConfig:
    # ---- Data Paths (Using your cleaned image dataset) ----
    TRAIN_PATH = '../data/training_data/model_splits/image/train.csv'
    VAL_PATH = '../data/training_data/model_splits/image/val.csv'
    TEST_PATH = '../data/training_data/model_splits/image/test.csv'
    
    # ---- Column Names (Your cleaned data format) ----
    IMAGE_PATH_COL = 'image_path'
    CATEGORY_COL = 'category'  # Main categories: education, insurance, legal, resume_employment
    SUBTYPE_COL = 'subtype'    # Specific document types
    
    # ---- M4-Optimized ViT Model & Training ----
    MODEL_NAME = 'google/vit-base-patch16-224'
    IMAGE_SIZE = 224           # Standard ViT input size
    BATCH_SIZE = 16            # Larger batch for images on M4
    GRADIENT_ACCUMULATION = 2  # Effective batch size = 16 * 2 = 32
    LR = 2e-5
    EPOCHS = 10                # More epochs for vision tasks
    WARMUP_RATIO = 0.1
    PATIENCE = 5
    WEIGHT_DECAY = 0.01
    
    # ---- M4 Performance Settings for Images ----
    NUM_WORKERS = 6            # Higher for image loading on M4
    PIN_MEMORY = True
    PREFETCH_FACTOR = 3        # Higher prefetch for images
    GRADIENT_CLIP = 1.0
    
    # ---- Data Augmentation ----
    USE_AUGMENTATION = True
    ROTATION_DEGREES = 10
    COLOR_JITTER = 0.1
    
    # ---- Output ----
    OUTPUT_DIR = 'artifacts_vit_m4_mvp'
    
config = M4ViTConfig()

print("🔧 M4-Optimized ViT Configuration:")
print(f"  📊 Dataset: {config.TRAIN_PATH}")
print(f"  🎯 Categories: {config.CATEGORY_COL}")
print(f"  🖼️  Image Size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"  🔢 Batch Size: {config.BATCH_SIZE} (x{config.GRADIENT_ACCUMULATION} = {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION})")
print(f"  🎨 Augmentation: {config.USE_AUGMENTATION}")
print(f"  ⚡ Workers: {config.NUM_WORKERS}")
print(f"  📱 Device: {DEVICE}")

os.makedirs(config.OUTPUT_DIR, exist_ok=True)


In [ ]:
# Load and analyze image dataset
print("📊 Loading Authenticator.ai image dataset...")

# Load the pre-split data
train_df = pd.read_csv(config.TRAIN_PATH)
val_df = pd.read_csv(config.VAL_PATH)  
test_df = pd.read_csv(config.TEST_PATH)

print(f"📈 Dataset Statistics:")
print(f"  Train: {len(train_df)} samples")
print(f"  Validation: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)} samples")

# Analyze categories and subtypes
print(f"\n🏷️  Category Distribution:")
category_counts = train_df[config.CATEGORY_COL].value_counts()
print(category_counts)

print(f"\n📋 Document Subtypes:")
subtype_counts = train_df[config.SUBTYPE_COL].value_counts()
print(f"Total subtypes: {len(subtype_counts)}")
print(subtype_counts)

# Encode categories for classification
category_encoder = LabelEncoder()
train_df['label'] = category_encoder.fit_transform(train_df[config.CATEGORY_COL])
val_df['label'] = category_encoder.transform(val_df[config.CATEGORY_COL])
test_df['label'] = category_encoder.transform(test_df[config.CATEGORY_COL])

# Save label mapping
label_map = {int(i): cls for i, cls in enumerate(category_encoder.classes_)}
with open(f'{config.OUTPUT_DIR}/category_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)

num_labels = len(category_encoder.classes_)
print(f"\n🎯 MVP Classification:")
print(f"Categories: {list(category_encoder.classes_)}")
print(f"Number of labels: {num_labels}")

print(f"\n📄 Sample Data:")
print(train_df[[config.IMAGE_PATH_COL, config.CATEGORY_COL, config.SUBTYPE_COL, 'label']].head(3))

train_df.head()


In [ ]:
# M4-Optimized Image Dataset and DataLoaders
print("🖼️  Creating M4-optimized image datasets...")

# Initialize ViT processor
processor = ViTImageProcessor.from_pretrained(config.MODEL_NAME)

class M4OptimizedImageDataset(Dataset):
    """M4-optimized image dataset with efficient transforms"""
    
    def __init__(self, df, processor, image_col='image_path', augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.image_col = image_col
        self.augment = augment
        
        # M4-optimized transforms
        if augment:
            self.transform = transforms.Compose([
                transforms.RandomRotation(config.ROTATION_DEGREES),
                transforms.ColorJitter(brightness=config.COLOR_JITTER, contrast=config.COLOR_JITTER),
                transforms.RandomHorizontalFlip(p=0.3),
            ])
        else:
            self.transform = None
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row[self.image_col]
        
        try:
            # Load image efficiently
            image = Image.open(image_path).convert('RGB')
            
            # Apply augmentation if training
            if self.transform:
                image = self.transform(image)
            
            # Process with ViT processor
            inputs = self.processor(image, return_tensors='pt')
            pixel_values = inputs['pixel_values'].squeeze(0)
            
            return {
                'pixel_values': pixel_values,
                'labels': torch.tensor(int(row['label']), dtype=torch.long)
            }
            
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            # Return a blank image as fallback
            blank_image = Image.new('RGB', (224, 224), color='white')
            inputs = self.processor(blank_image, return_tensors='pt')
            return {
                'pixel_values': inputs['pixel_values'].squeeze(0),
                'labels': torch.tensor(int(row['label']), dtype=torch.long)
            }

# Create M4-optimized datasets
train_ds = M4OptimizedImageDataset(train_df, processor, config.IMAGE_PATH_COL, augment=config.USE_AUGMENTATION)
val_ds = M4OptimizedImageDataset(val_df, processor, config.IMAGE_PATH_COL, augment=False)
test_ds = M4OptimizedImageDataset(test_df, processor, config.IMAGE_PATH_COL, augment=False)

# M4-optimized data loaders
train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
    prefetch_factor=config.PREFETCH_FACTOR,
    persistent_workers=True
)

print(f"✅ M4-Optimized Image Datasets Created:")
print(f"  Train: {len(train_ds)} samples ({len(train_loader)} batches)")
print(f"  Val: {len(val_ds)} samples ({len(val_loader)} batches)")
print(f"  Test: {len(test_ds)} samples ({len(test_loader)} batches)")
print(f"  Image size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print(f"  Augmentation: {config.USE_AUGMENTATION}")
print(f"  Workers: {config.NUM_WORKERS}")

len(train_ds), len(val_ds), len(test_ds)


In [ ]:
# M4-Optimized ViT Model
print("🤖 Initializing M4-optimized Vision Transformer...")

# Load pre-trained ViT model
model = ViTForImageClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

# Move to M4 device
model = model.to(DEVICE)

# Model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 M4 ViT Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / (1024**2):.1f} MB")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Categories: {num_labels}")
print(f"  Image size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")

model


In [ ]:
# M4-Optimized Training Setup for ViT
print("⚙️ Setting up M4-optimized ViT training...")

# Calculate class weights for balanced training
cnt = Counter(train_df['label'])
total = sum(cnt.values())
class_weights = torch.tensor([total/(num_labels*cnt[i]) for i in range(num_labels)], dtype=torch.float32).to(DEVICE)
print(f'📊 Class distribution: {dict(cnt)}')
print(f'⚖️  Class weights: {class_weights.cpu().numpy()}')

# M4-optimized optimizer for ViT
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.LR,
    weight_decay=config.WEIGHT_DECAY,
    eps=1e-8,
    betas=(0.9, 0.999)
)

# Calculate training steps with gradient accumulation
steps_per_epoch = len(train_loader)
total_training_steps = steps_per_epoch * config.EPOCHS
num_warmup_steps = int(config.WARMUP_RATIO * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps,
    total_training_steps
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

print(f"🔧 ViT Training Configuration:")
print(f"  Steps per epoch: {steps_per_epoch}")
print(f"  Total training steps: {total_training_steps}")
print(f"  Warmup steps: {num_warmup_steps}")
print(f"  Gradient accumulation: {config.GRADIENT_ACCUMULATION}")
print(f"  Effective batch size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")


In [ ]:
# M4-Optimized ViT Training Loop with Progress Tracking
from tqdm.auto import tqdm
import time

def run_vit_epoch_with_progress(dataloader, training=True, epoch_num=1):
    """M4-optimized ViT training/validation with detailed progress tracking"""
    model.train() if training else model.eval()
    
    phase = "Training" if training else "Validation"
    losses, all_preds, all_labels = [], [], []
    
    # Progress bar with detailed info
    pbar = tqdm(dataloader, 
                desc=f"Epoch {epoch_num} - ViT {phase}", 
                leave=True,
                dynamic_ncols=True)
    
    batch_count = 0
    accumulated_loss = 0
    
    # For gradient accumulation
    if training:
        optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        batch_count += 1
        
        # Move batch to device
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        
        with torch.set_grad_enabled(training):
            # Forward pass
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss
            
            if training:
                # Scale loss for gradient accumulation
                loss = loss / config.GRADIENT_ACCUMULATION
                loss.backward()
                accumulated_loss += loss.item()
                
                # Gradient accumulation step
                if (batch_idx + 1) % config.GRADIENT_ACCUMULATION == 0 or (batch_idx + 1) == len(dataloader):
                    # Gradient clipping for stability
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    
                    # Update progress with accumulated loss
                    pbar.set_postfix({
                        'Loss': f'{accumulated_loss:.4f}',
                        'Batch': f'{batch_count}/{len(dataloader)}',
                        'LR': f'{scheduler.get_last_lr()[0]:.2e}',
                        'Device': DEVICE
                    })
                    accumulated_loss = 0
            else:
                # Validation mode
                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Batch': f'{batch_count}/{len(dataloader)}',
                    'Device': DEVICE
                })
        
        # Collect predictions and labels
        with torch.no_grad():
            logits = outputs.logits
            preds = logits.argmax(dim=-1).detach().cpu().numpy()
            lbls = labels.detach().cpu().numpy()
            
            losses.append(loss.item() * (config.GRADIENT_ACCUMULATION if training else 1))
            all_preds.extend(list(preds))
            all_labels.extend(list(lbls))
        
        # Clear cache periodically for M4 memory management
        if batch_count % 30 == 0 and DEVICE == 'mps':  # More frequent for images
            torch.backends.mps.empty_cache()
    
    # Calculate metrics
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    f1m = f1_score(all_labels, all_preds, average='macro')
    
    return avg_loss, acc, f1m

# M4-Optimized ViT Training Loop
print("🚀 Starting M4-Optimized ViT Training...")
print(f"📊 Training on {len(train_ds)} images, validating on {len(val_ds)} images")
print(f"🎯 Categories: {list(category_encoder.classes_)}")
print(f"🖼️  Image size: {config.IMAGE_SIZE}x{config.IMAGE_SIZE}")
print("=" * 80)

best_f1, patience_counter = -1.0, 0
training_history = []
start_time = time.time()

for epoch in range(1, config.EPOCHS + 1):
    epoch_start = time.time()
    
    print(f"\n🔄 EPOCH {epoch}/{config.EPOCHS}")
    print(f"⏰ Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Training phase
    tr_loss, tr_acc, tr_f1 = run_vit_epoch_with_progress(train_loader, training=True, epoch_num=epoch)
    
    # Validation phase
    va_loss, va_acc, va_f1 = run_vit_epoch_with_progress(val_loader, training=False, epoch_num=epoch)
    
    # Calculate epoch time
    epoch_time = time.time() - epoch_start
    
    # Store history
    epoch_stats = {
        'epoch': epoch,
        'train_loss': tr_loss,
        'val_loss': va_loss,
        'train_acc': tr_acc,
        'val_acc': va_acc,
        'train_f1': tr_f1,
        'val_f1': va_f1,
        'epoch_time': epoch_time
    }
    training_history.append(epoch_stats)
    
    # Print epoch summary
    print(f"\n📈 EPOCH {epoch} SUMMARY:")
    print(f"  🏃 Train: Loss={tr_loss:.4f}, Acc={tr_acc:.4f}, F1={tr_f1:.4f}")
    print(f"  ✅ Val:   Loss={va_loss:.4f}, Acc={va_acc:.4f}, F1={va_f1:.4f}")
    print(f"  ⏱️  Time: {epoch_time:.1f}s")
    
    # Early stopping and model saving
    if va_f1 > best_f1:
        best_f1 = va_f1
        patience_counter = 0
        
        # Save best model
        print(f"  🎉 New best F1: {best_f1:.4f} - Saving model!")
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        torch.save(model.state_dict(), f'{config.OUTPUT_DIR}/best_model.pt')
        
        # Save training history
        with open(f'{config.OUTPUT_DIR}/training_history.json', 'w') as f:
            json.dump(training_history, f, indent=2)
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement - Patience: {patience_counter}/{config.PATIENCE}")
        
        if patience_counter >= config.PATIENCE:
            print(f"\n🛑 Early stopping triggered after {epoch} epochs!")
            break

total_time = time.time() - start_time
print(f"\n🏁 ViT TRAINING COMPLETED!")
print(f"  🏆 Best Validation F1: {best_f1:.4f}")
print(f"  ⏱️  Total Time: {total_time/60:.1f} minutes")
print(f"  💾 Model saved to: {config.OUTPUT_DIR}/best_model.pt")


In [ ]:
# M4-Optimized ViT Test Evaluation
print("🧪 Loading best ViT model for test evaluation...")
model.load_state_dict(torch.load(f'{config.OUTPUT_DIR}/best_model.pt', map_location=DEVICE))
model.to(DEVICE)
model.eval()

print("🔍 Running ViT test evaluation with progress tracking...")
all_preds, all_labels = [], []

# Test evaluation with progress bar
test_pbar = tqdm(test_loader, desc="ViT Test Evaluation", leave=True)
for batch in test_pbar:
    pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
    labels = batch['labels'].to(DEVICE, non_blocking=True)
    
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values, labels=labels)
        logits = outputs.logits
        
        preds = logits.argmax(dim=-1).detach().cpu().numpy()
        lbls = labels.detach().cpu().numpy()
        
        all_preds.extend(list(preds))
        all_labels.extend(list(lbls))
    
    test_pbar.set_postfix({'Samples': len(all_preds)})

# Calculate final metrics
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"\n🏆 FINAL ViT TEST RESULTS:")
print(f"  📊 Test Accuracy: {test_acc:.4f}")
print(f"  📈 Test F1 (Macro): {test_f1:.4f}")
print(f"  📈 Test F1 (Weighted): {test_f1_weighted:.4f}")
print(f"  📝 Test Samples: {len(all_labels)}")

print(f"\n📋 DETAILED CLASSIFICATION REPORT:")
print("=" * 60)
target_names = [category_encoder.classes_[i] for i in range(len(category_encoder.classes_))]
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

# Save test results
test_results = {
    'test_accuracy': test_acc,
    'test_f1_macro': test_f1,
    'test_f1_weighted': test_f1_weighted,
    'test_samples': len(all_labels),
    'categories': list(category_encoder.classes_),
    'classification_report': classification_report(all_labels, all_preds, target_names=target_names, digits=4, output_dict=True)
}

with open(f'{config.OUTPUT_DIR}/test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n💾 ViT Results saved to: {config.OUTPUT_DIR}/test_results.json")
